# GAVE2 V13 R5.1: Fine Calibration for a 7.7 Release Gate

This is the repaired three-fold V13 experiment targeting an official score of
`7.7`. It keeps every image on its native `1536 x 1024` canvas. Training and
raw predictions still resume from the existing V13 run, but final selection is
anchored to the officially stronger V12 output. The rejected geodesic growth
stage is replaced by prune-only calibration with protected teacher paths. R5.1
fine-searches the upper calibration boundary found by R5 and permits each Task
3 target to pass or freeze independently.

The target is not a promise. The notebook produces one uploadable ZIP only
when both segmentation tasks, the registered-FFA Task 3 model, and the final
score projection pass strict gates. The final ZIP must also be strictly below
`100,000,000` bytes while preserving every Task 1/2 threshold decision and
every Task 3 byte. Otherwise it records `DO_NOT_SUBMIT` and disconnects.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DRIVE_BASE = Path("/content/drive/MyDrive/MICCAI2026")
ARCHIVE_PATH = DRIVE_BASE / "miccai_v13.zip"
WORK_ROOT = Path("/content/MICCAI2026")
DATA_ROOT = WORK_ROOT / "GAVE2_preliminary"
RUN_DIR = DRIVE_BASE / "runs/gave2_v13_channel_path_3fold"
V8_RUN_DIR = DRIVE_BASE / "runs/gave2_r2v2_v8"
V12_RUN_DIR = DRIVE_BASE / "runs/gave2_v12_safe_3fold"
ASSET_DIR = DRIVE_BASE / "assets"
SUBMISSION_ROOT = DRIVE_BASE / "submissions/gave2_v13_r51_topology_safe"

TEAM_ID = "梯度不下降队"
RUN_MINIMA = True
AUTO_DISCONNECT = True
RELEASE_TARGET = 7.7
PORTAL_MAXIMUM_BYTES = 100_000_000
MINIMA_MIN_SUCCESS_FRACTION = 0.90
EXPECTED_RUNTIME_BUILD_ID = "gave2-v13-r6-r51-fine-calibration"

assert ARCHIVE_PATH.is_file(), ARCHIVE_PATH
RUN_DIR.mkdir(parents=True, exist_ok=True)
print({"archive": str(ARCHIVE_PATH), "run_dir": str(RUN_DIR), "team_id": TEAM_ID})

## Verify and extract the immutable V13 runtime

In [ ]:
import hashlib
import json
import os
import shutil
import zipfile

def bytes_sha256(payload):
    return hashlib.sha256(payload).hexdigest()

with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    assert archive.testzip() is None
    manifest = json.loads(archive.read("archive_manifest.json"))
    if manifest.get("runtime_build_id") != EXPECTED_RUNTIME_BUILD_ID:
        raise RuntimeError(
            "Runtime ZIP does not match this notebook. "
            f"Expected {EXPECTED_RUNTIME_BUILD_ID!r}, found {manifest.get('runtime_build_id')!r}. "
            "Replace MyDrive/MICCAI2026/miccai_v13.zip."
        )
    names = set(archive.namelist())
    assert names == set(manifest["members"]) | {"archive_manifest.json"}
    required = {
        "GAVE2_preliminary/training/images/g_001.png",
        "GAVE2_preliminary/validation/images/g_051.png",
        "experiments/gave2_v13/train.py",
        "experiments/gave2_v13/selection.py",
        "experiments/gave2_v13/task3.py",
        "experiments/gave2_v13/release.py",
        "experiments/gave2_v13/compact.py",
        "experiments/gave2_v12/minima_adapter.py",
        "experiments/gave2_v8/predict_r2v2.py",
        "tests/gave2_v13/test_selection.py",
        "tests/gave2_v13/test_compact_submission.py",
        "tests/gave2_v13/test_losses.py",
        "submission/GAVE2_R2V2_FFA_Residual_V12_Colab.ipynb",
    }
    assert required.issubset(names), sorted(required - names)
    for member, expected in manifest["sha256"].items():
        assert bytes_sha256(archive.read(member)) == expected, member
    if WORK_ROOT.exists():
        shutil.rmtree(WORK_ROOT)
    WORK_ROOT.mkdir(parents=True)
    archive.extractall(WORK_ROOT)

os.chdir(WORK_ROOT)
assert DATA_ROOT.is_dir()
print({"verified_members": len(manifest["members"]), "runtime_build_id": manifest["runtime_build_id"]})

## Install pinned dependencies and run the release tests

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "experiments/gave2_v13/requirements.txt"],
    check=True,
)
import torch
import kornia

assert torch.cuda.is_available(), "Select a GPU runtime"
assert torch.cuda.is_bf16_supported(), "V13 requires a BF16-capable CUDA GPU"
test_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/gave2_v13",
        "tests/gave2_v12",
        "tests/gave2_v8/test_store_and_fusion.py",
        "tests/gave2_v8/test_submission.py",
        "-q",
    ],
    cwd=WORK_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(test_result.stdout, flush=True)
test_result.check_returncode()
gpu = torch.cuda.get_device_properties(0)
print({
    "torch": torch.__version__,
    "kornia": kornia.__version__,
    "gpu": gpu.name,
    "vram_gib": round(gpu.total_memory / 1024**3, 2),
    "bf16": True,
})

In [ ]:
def run_module(module, *arguments, check=True):
    command = [sys.executable, "-m", module, *map(str, arguments)]
    print("RUN:", " ".join(command), flush=True)
    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    process = subprocess.Popen(
        command,
        cwd=WORK_ROOT,
        env=environment,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    tail = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        tail.append(line.rstrip())
        tail = tail[-100:]
    return_code = process.wait()
    result = subprocess.CompletedProcess(command, return_code, stdout="\n".join(tail))
    if check and return_code:
        raise RuntimeError(
            f"{module} failed with return code {return_code}. Last output:\n" + "\n".join(tail)
        )
    return result

def stop_and_disconnect(reason):
    decision = {"version": 13, "status": "DO_NOT_SUBMIT", "reasons": [str(reason)]}
    (RUN_DIR / "release_decision_r51.json").write_text(json.dumps(decision, indent=2))
    print(json.dumps(decision, indent=2), flush=True)
    if AUTO_DISCONNECT:
        from google.colab import runtime
        runtime.unassign()
    raise SystemExit(reason)

R2V2_SOURCE = WORK_ROOT / "external/R2-V2"
R2V2_WEIGHTS = ASSET_DIR / "r2v2"
V8_PREDICTIONS = V8_RUN_DIR / "predictions"
V12_SELECTED = V12_RUN_DIR / "predictions/selected"
PREPARED_ROOT = RUN_DIR / "prepared_ffa"
MATCHES_ROOT = RUN_DIR / "minima_matches"
FOLD_MANIFEST = RUN_DIR / "fold_manifest.json"
TASK3_RUN_DIR = RUN_DIR / "r51"

## Acquire the pinned R2-V2 teacher

In [ ]:
run_module(
    "experiments.gave2_v8.assets",
    "--source-dir", R2V2_SOURCE,
    "--weights-dir", R2V2_WEIGHTS,
)

for split in ("training", "validation"):
    for model_type in ("av", "bv"):
        run_module(
            "experiments.gave2_v8.predict_r2v2",
            "--data-root", DATA_ROOT,
            "--source-dir", R2V2_SOURCE,
            "--weights-dir", R2V2_WEIGHTS,
            "--output-store", V8_PREDICTIONS / split / model_type,
            "--model-type", model_type,
            "--split", split,
            "--amp", "bf16",
            "--tta",
        )
    run_module(
        "experiments.gave2_v8.fuse",
        "--av-store", V8_PREDICTIONS / split / "av",
        "--bv-store", V8_PREDICTIONS / split / "bv",
        "--output-store", V8_PREDICTIONS / split / "direct",
        "--split", split,
    )

## Register FFA to CFP using MINIMA with geometric QA

In [ ]:
MINIMA_SOURCE = WORK_ROOT / "external/MINIMA"
MINIMA_CHECKPOINT = ASSET_DIR / "minima/minima_loftr.ckpt"

if RUN_MINIMA:
    run_module(
        "experiments.gave2_v12.assets",
        "--source-dir", MINIMA_SOURCE,
        "--checkpoint", MINIMA_CHECKPOINT,
    )
    run_module(
        "experiments.gave2_v12.minima_adapter",
        "--data-root", DATA_ROOT,
        "--source-dir", MINIMA_SOURCE,
        "--checkpoint", MINIMA_CHECKPOINT,
        "--output-root", MATCHES_ROOT,
        "--split", "training",
        "--phase", "FFA_A",
        "--threshold", "0.20",
        "--failure-policy", "error",
        "--limit-cases", "1",
    )
    failures = []
    for split in ("training", "validation"):
        for phase in ("FFA_A", "FFA_AV"):
            run_module(
                "experiments.gave2_v12.minima_adapter",
                "--data-root", DATA_ROOT,
                "--source-dir", MINIMA_SOURCE,
                "--checkpoint", MINIMA_CHECKPOINT,
                "--output-root", MATCHES_ROOT,
                "--split", split,
                "--phase", phase,
                "--threshold", "0.20",
                "--failure-policy", "identity",
            )
            summary = json.loads((MATCHES_ROOT / split / phase / "summary.json").read_text())
            success = summary["successful_cases"] / max(summary["cases"], 1)
            print({"split": split, "phase": phase, "successful_match_fraction": round(success, 3)})
            if success < MINIMA_MIN_SUCCESS_FRACTION:
                failures.append(f"{split}/{phase}: {success:.1%}")
    if failures:
        stop_and_disconnect("MINIMA extraction gate failed: " + "; ".join(failures))

for split in ("training", "validation"):
    run_module(
        "experiments.gave2_v12.prepare",
        "--data-root", DATA_ROOT,
        "--matches-root", MATCHES_ROOT,
        "--output-root", PREPARED_ROOT,
        "--split", split,
        "--fallback", "identity",
    )
    summary = json.loads((PREPARED_ROOT / split / "summary.json").read_text())
    print({"split": split, "registration_acceptance": summary["acceptance_fraction"]})

## Create the immutable balanced three-fold split

In [ ]:
run_module(
    "experiments.gave2_v12.folds",
    "--data-root", DATA_ROOT,
    "--output", FOLD_MANIFEST,
    "--seed", 77,
)
folds = json.loads(FOLD_MANIFEST.read_text())
assert len(folds["folds"]) == 3
print([len(fold["validation"]) for fold in folds["folds"]])

## Select the largest full-canvas profile that fits this BF16 GPU

In [ ]:
PROFILE_CANDIDATES = (
    (24, 2, False),
    (20, 3, False),
    (20, 2, False),
    (16, 3, False),
    (16, 2, False),
    (16, 1, True),
    (12, 1, True),
)

def select_profile(task):
    for base_channels, batch_size, checkpointing in PROFILE_CANDIDATES:
        arguments = [
            "--data-root", DATA_ROOT,
            "--teacher-store", V8_PREDICTIONS / "training/direct",
            "--task", task,
            "--base-channels", base_channels,
            "--batch-size", batch_size,
            "--steps", 2,
            "--amp", "bf16",
            "--activation-checkpointing" if checkpointing else "--no-activation-checkpointing",
        ]
        if task == "task2":
            arguments += ["--prepared-root", PREPARED_ROOT]
        result = run_module("experiments.gave2_v13.memory_test", *arguments, check=False)
        if result.returncode == 0:
            return {
                "base_channels": base_channels,
                "batch_size": batch_size,
                "activation_checkpointing": checkpointing,
            }
        failure = (result.stdout or "").lower()
        if not any(marker in failure for marker in ("out of memory", "outofmemoryerror", "cublas_status_alloc_failed")):
            raise RuntimeError(f"{task} profile failed for a non-memory reason; stop immediately")
        torch.cuda.empty_cache()
    raise RuntimeError(f"No full-canvas BF16 profile fits {task}")

PROFILES = {"task2": select_profile("task2"), "task1": select_profile("task1")}
(RUN_DIR / "selected_profiles.json").write_text(json.dumps(PROFILES, indent=2))
print(PROFILES)

## Train Task 2 first, then Task 1; all folds resume automatically

In [ ]:
def train_task(task, epochs, minimum_epochs):
    profile = PROFILES[task]
    for fold in range(3):
        fold_dir = RUN_DIR / "models" / task / f"fold_{fold}"
        arguments = [
            "--data-root", DATA_ROOT,
            "--run-dir", RUN_DIR,
            "--fold-manifest", FOLD_MANIFEST,
            "--teacher-store", V8_PREDICTIONS / "training/direct",
            "--task", task,
            "--fold", fold,
            "--base-channels", profile["base_channels"],
            "--batch-size", profile["batch_size"],
            "--workers", 2,
            "--epochs", epochs,
            "--minimum-epochs", minimum_epochs,
            "--early-stopping-patience", 7,
            "--min-delta", 0.003,
            "--lr", 0.0002,
            "--amp", "bf16",
            "--seed", 77,
            "--activation-checkpointing" if profile["activation_checkpointing"] else "--no-activation-checkpointing",
        ]
        if task == "task2":
            arguments += ["--prepared-root", PREPARED_ROOT]
        if (fold_dir / "config.json").exists():
            arguments.append("--resume")
        run_module("experiments.gave2_v13.train", *arguments)

train_task("task2", epochs=80, minimum_epochs=25)
train_task("task1", epochs=55, minimum_epochs=20)

## Generate three-fold OOF and validation predictions

In [ ]:
RAW_ROOT = RUN_DIR / "predictions/raw"

for task in ("task2", "task1"):
    for split in ("training", "validation"):
        arguments = [
            "--data-root", DATA_ROOT,
            "--run-dir", RUN_DIR,
            "--fold-manifest", FOLD_MANIFEST,
            "--teacher-store", V8_PREDICTIONS / split / "direct",
            "--output-store", RAW_ROOT / split / task,
            "--task", task,
            "--split", split,
            "--amp", "bf16",
            "--tta",
        ]
        if task == "task2":
            arguments += ["--prepared-root", PREPARED_ROOT]
        run_module("experiments.gave2_v13.predict", *arguments)

## Repair selection against V12 using protected paths

In [ ]:
SELECTION_ROOT = RUN_DIR / "selection_r51"
SELECTED_ROOT = RUN_DIR / "predictions/selected_r51"
SELECTION_ROOT.mkdir(parents=True, exist_ok=True)

for task in ("task2", "task1"):
    for split in ("training", "validation"):
        teacher_manifest = V12_SELECTED / split / task / "completion_manifest.json"
        if not teacher_manifest.exists():
            stop_and_disconnect(
                f"Missing V12 selected teacher for {task}/{split}: {teacher_manifest}. "
                "Keep V12; do not fall back to V8 for the R5.1 selector."
            )
    selection_path = SELECTION_ROOT / f"{task}.json"
    run_module(
        "experiments.gave2_v13.selection", "search",
        "--data-root", DATA_ROOT,
        "--fold-manifest", FOLD_MANIFEST,
        "--teacher-store", V12_SELECTED / "training" / task,
        "--raw-store", RAW_ROOT / "training" / task,
        "--output-config", selection_path,
        "--task", task,
        "--search-mode", "topology_safe",
        "--corridor-radius", 2,
        "--alpha-values", 0.85, 1.0,
        "--decision-threshold-values", 0.575, 0.585, 0.595, 0.605, 0.615, 0.625,
        "--temperature-values", 0.85, 0.90, 0.95,
        "--minimum-score-gain", 0.10,
        "--minimum-fold-score-gain", 0.02,
        "--minimum-topology-gain", 0.005,
        "--minimum-fold-topology-gain", -0.002,
        "--maximum-sensitivity-drop", 0.025,
        "--maximum-fold-sensitivity-drop", 0.04,
        "--maximum-reassignment-fraction", 0.0,
        "--path-shortlist", 5,
        "--paths-per-case", 100,
        "--path-seed", 77,
    )
    selection = json.loads(selection_path.read_text())
    print(task, json.dumps(selection["selected"], indent=2))
    if not selection["accepted"]:
        stop_and_disconnect(f"{task} failed the repaired sampled COR/INF OOF gate; keep V12")
    for split in ("training", "validation"):
        run_module(
            "experiments.gave2_v13.selection", "apply",
            "--data-root", DATA_ROOT,
            "--teacher-store", V12_SELECTED / split / task,
            "--raw-store", RAW_ROOT / split / task,
            "--output-store", SELECTED_ROOT / split / task,
            "--selection", selection_path,
            "--task", task,
            "--split", split,
        )

## Audit registered-FFA Task 3 correction

In [ ]:
V8_TASK3_SOURCE = WORK_ROOT / "experiments/gave2_v8/assets/proven_task3"
run_module(
    "experiments.gave2_v13.task3",
    "--data-root", DATA_ROOT,
    "--run-dir", TASK3_RUN_DIR,
    "--training-store", SELECTED_ROOT / "training/task2",
    "--validation-store", SELECTED_ROOT / "validation/task2",
    "--prepared-root", PREPARED_ROOT,
    "--v8-task3-source", V8_TASK3_SOURCE,
    "--outer-repeats", 10,
    "--minimum-gain", 0.12,
    "--minimum-selection-rate", 0.80,
    "--minimum-bootstrap-probability", 0.95,
    "--maximum-mean-shift", 1.0,
    "--maximum-abs-z", 8.0,
    "--seed", 77,
)
TASK3_AUDIT = TASK3_RUN_DIR / "task3/audit.json"
task3 = json.loads(TASK3_AUDIT.read_text())
required_task3 = {"vein_density"}
if not required_task3.issubset(task3["accepted_targets"]):
    stop_and_disconnect(
        "Task 3 did not robustly accept vein_density: " + str(task3["accepted_targets"])
    )
print({
    "accepted_task3": task3["accepted_targets"],
    "frozen_task3": task3["frozen_targets"],
    "note": "Rejected targets remain byte-for-byte frozen to the proven source.",
})

## Build and certify the one V13 candidate

In [ ]:
run_module(
    "experiments.gave2_v13.submission",
    "--data-root", DATA_ROOT,
    "--output-root", SUBMISSION_ROOT,
    "--team-id", TEAM_ID,
    "--task1-store", SELECTED_ROOT / "validation/task1",
    "--task2-store", SELECTED_ROOT / "validation/task2",
    "--task3-source", TASK3_RUN_DIR / "task3/validation",
    "--task1-selection", SELECTION_ROOT / "task1.json",
    "--task2-selection", SELECTION_ROOT / "task2.json",
    "--task3-audit", TASK3_AUDIT,
)
SUBMISSION_MANIFEST = SUBMISSION_ROOT / "v13_candidate/manifest.json"
submission = json.loads(SUBMISSION_MANIFEST.read_text())
assert 0 < submission["zip_bytes"] < PORTAL_MAXIMUM_BYTES
assert submission["maximum_submission_bytes"] == PORTAL_MAXIMUM_BYTES
assert submission["threshold_mismatch_pixels"] == 0
assert submission["task3_byte_identical"] is True
print({
    "upload_zip": submission["zip"],
    "size_mb": round(submission["zip_bytes"] / 1_000_000, 3),
    "headroom_mb": round(submission["headroom_bytes"] / 1_000_000, 3),
    "probability_bits": submission["probability_bits"],
})
print(json.dumps(submission, indent=2, ensure_ascii=False))

## Final release decision, source package, and automatic disconnect

In [ ]:
DECISION_PATH = RUN_DIR / "release_decision_r51.json"
release_result = run_module(
    "experiments.gave2_v13.release",
    "--task1-selection", SELECTION_ROOT / "task1.json",
    "--task2-selection", SELECTION_ROOT / "task2.json",
    "--task3-audit", TASK3_AUDIT,
    "--submission-manifest", SUBMISSION_MANIFEST,
    "--output", DECISION_PATH,
    "--release-target", RELEASE_TARGET,
    "--minimum-local-task-score", 8.7,
    "--segmentation-transfer-scale", 1.0,
    "--required-task3-targets", "vein_density",
    check=False,
)
decision = json.loads(DECISION_PATH.read_text())
print(json.dumps(decision, indent=2, ensure_ascii=False))

if decision["status"] == "READY_FOR_ONE_CAUTIOUS_SUBMISSION":
    assert decision["zip_valid"] and decision["compact_valid"]
    assert 0 < decision["zip_bytes"] < PORTAL_MAXIMUM_BYTES
    SOURCE_ZIP = SUBMISSION_ROOT / "GAVE2_V13_source_code.zip"
    subprocess.run(
        [
            sys.executable,
            "scripts/build_miccai_v13_archive.py",
            "--output", SOURCE_ZIP,
            "--code-only",
            "--force",
        ],
        cwd=WORK_ROOT,
        check=True,
    )
    print("Submit exactly one ZIP:", decision["zip"])
    print("Competition ZIP size (MB):", round(decision["zip_bytes"] / 1_000_000, 3))
    print("Portal headroom (MB):", round(decision["headroom_bytes"] / 1_000_000, 3))
    print("Organizer source package:", SOURCE_ZIP)
else:
    print("DO NOT SUBMIT V13 R5.1. Keep the V12 leaderboard result.")

if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()